# **Creating Iterative Base Class with OOPs Generelization**

In [131]:
import numpy as np
import pandas as pd
import yfinance as yf

In [132]:
class IterativeBase():
  def __init__(self,companies,period,interval,amount,use_spread=True):
    self.companies=companies
    self.period=period
    self.interval=interval
    self.initial_balance=amount
    self.current_balance=amount
    self.use_spread=use_spread
    self.units = 0
    self.trades = 0
    self.get_data()

  def get_data(self):
    self.data=yf.download(self.companies,period=self.period,interval=self.interval)
    self.df=pd.DataFrame(self.data["Close"])
    self.spread_df=pd.DataFrame(self.data["High"]-self.data["Low"])
    for i in self.companies:
      self.df[f"LR {i}"]=np.log(self.df[i]/self.df[i].shift(1))
      self.df[f"Spread {i}"]=self.spread_df[i]
    self.df.dropna(inplace=True)
    return self.df

  # returns values of date, price, spread for only one stock.
  def get_values(self,asset,bar):   # Need to modify for asset=[stock lists]
    if asset in self.companies: # Check if the asset is in self.companies or not
      date=str(self.df.index[bar].date())
      price=round(self.df[asset].iloc[bar],5)    # stores the value of share price at the given bar.
      spread=round(self.df[f"Spread {asset}"].iloc[bar],5)   # stores the spread of share at the given bar.
      return date,price,spread
    else:
      print(f"{asset} is not valid ticker name. Try from {self.companies}")

  # class to buy instruments
  def buy_instrument(self,asset,bar,units=None,amount=None):
    date,price,spread=self.get_values(asset,bar)
    if self.use_spread:
      price+=spread/2    # Buy Price based on the spread
    if amount!=None:
      units=int(amount/price)
    self.current_balance-=units*price
    self.units+=units
    self.trades+=1
    print(f"{date} | Buying {units} shares of {asset} at {round(price,3)}")
    print(f"Current Balance = {self.current_balance}")

  # class to sell instruments
  def sell_instrument(self,asset,bar,units=None,amount=None):
    date,price,spread=self.get_values(asset,bar)
    if self.use_spread:
      price+=spread/2   # Sell price based on the spread
    if amount!=None:
      units=int((amount/price)-1)
    self.current_balance+=units*price
    self.units-=units
    self.trades+=1
    print(f"{date} | Selling {units} shares of {asset} at {round(price,3)}")
    print(f"Current Balance = {self.current_balance}")

  # Class to find the current positon value
  def cpv(self,asset,bar):
    date,price,spread=self.get_values(asset,bar)
    cpv=self.units*price
    print(f"{date} | Current Position Value = {round(cpv,2)}")

  # Class to find Net Asset Value
  def nav(self,asset,bar):
    date,price,spread=self.get_values(asset,bar)
    nav=self.current_balance+self.units*price
    print(f"{date} | Net Assets Value = {round(nav,2)}")

  #Class to close position
  def close_position(self,asset,bar):
    date,price,spread=self.get_values(asset,bar)
    print(f"{date} | +++ CLOSING POSITION +++")
    self.units=0
    self.current_balance+=self.units*price
    self.current_balance-=abs(self.units*(spread/2)*self.use_spread)
    self.trades+=1
    print(f"{date} | Closing position of {self.units} at {price}")
    perf=((self.current_balance-self.initial_balance)/self.initial_balance)*100
    print(f"{date} | Current Balance = {round(self.current_balance,2)}")
    print(f"{date} | Net Performace = {round(perf,2)} %")
    print(f"{date} | Number of Trades Executed = {self.trades}")

In [133]:
a=IterativeBase(["HAL.NS",'BSE.NS','FORTIS.NS'],"1y","1d",14000)
a.get_data()

/tmp/ipython-input-3920854102.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  self.data=yf.download(self.companies,period=self.period,interval=self.interval)
[*********************100%***********************]  3 of 3 completed
/tmp/ipython-input-3920854102.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  self.data=yf.download(self.companies,period=self.period,interval=self.interval)
[*********************100%***********************]  3 of 3 completed


Ticker,BSE.NS,FORTIS.NS,HAL.NS,LR HAL.NS,Spread HAL.NS,LR BSE.NS,Spread BSE.NS,LR FORTIS.NS,Spread FORTIS.NS
Date,,,,,,,,,
2025-01-28,1713.512329,580.513489,3561.266846,-0.019671,209.323367,-0.053684,118.409869,-0.018327,18.977557
2025-01-29,1781.963135,627.807495,3661.550049,0.027770,131.055837,0.039170,82.523051,0.078321,53.286954
2025-01-30,1740.809692,646.984863,3739.372070,0.021031,90.784276,-0.023365,78.037203,0.030089,27.467516
2025-01-31,1763.089355,639.443787,3895.362793,0.040869,182.261133,0.012717,33.427896,-0.011724,26.069206
2025-02-01,1795.686523,626.109497,3735.216553,-0.041981,367.293112,0.018320,72.587715,-0.021073,27.567434
...,...,...,...,...,...,...,...,...,...
2026-01-19,2726.699951,892.049988,4504.299805,0.016926,102.899902,-0.029630,83.500000,-0.004920,10.350037
2026-01-20,2653.399902,856.450012,4355.200195,-0.033662,198.000000,-0.027250,92.000000,-0.040726,41.600037
2026-01-21,2628.800049,844.150024,4259.399902,-0.022242,102.600098,-0.009314,82.000000,-0.014466,25.450012


In [134]:
a.get_values("HAL.NS",7)

('2025-02-05', np.float64(3777.12085), np.float64(102.7075))

In [135]:
a.buy_instrument("BSE.NS",3,amount=12000)

2025-01-31 | Buying 6 shares of BSE.NS at 1779.803
Current Balance = 3321.1801400000004


In [136]:
a.sell_instrument("BSE.NS",3,amount=12000)

2025-01-31 | Selling 5 shares of BSE.NS at 1779.803
Current Balance = 12220.19669


In [137]:
a.nav("BSE.NS",3)

2025-01-31 | Net Assets Value = 13983.29


In [138]:
a.close_position("BSE.NS",3)

2025-01-31 | +++ CLOSING POSITION +++
2025-01-31 | Closing position of 0 at 1763.08936
2025-01-31 | Current Balance = 12220.2
2025-01-31 | Net Performace = -12.71 %
2025-01-31 | Number of Trades Executed = 3
